In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from openai import OpenAI
# from getpass import getpass: This would not display the api key you enter
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI()

In [ ]:
topics = [
    "State",
    "Streaming",
    "Checkpoints",
    "Human Approval",
    "Interrupt",
    "Time Travel",
    "Parallel Graph"
]

for topic in topics:
    print ("✅", topic)

In [ ]:
class AgentState(TypedDict):
    question: str
    draft: str
    review: str
    approval: bool
    final: str

In [ ]:
state = {
    "question": "Write banking email",
    "draft": "",
    "review": "",
    "approval": False,
    "final": ""
}

In [ ]:
for key, value in state.items():
    print(f"{key}: {value}")

In [ ]:
def writer(state):
    prompt ="""
    Write a professional email
    Request
    {state['question']}
    """
    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt
    )
    state['draft'] = response.output_text
    return state

new_state = writer(state)
new_state

In [ ]:
def reviewer(state):
    prompt = f"""
    Review the following email draft and provide feedback:
    Draft: {state['draft']}
    """
    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt
    )
    state['review'] = response.output_text
    return state

state = reviewer(new_state)
state['review']

In [ ]:
builder = StateGraph(AgentState)

builder.add_node("writer", writer)
builder.add_node("reviewer", reviewer)

builder.add_edge(START, "writer")
builder.add_edge("writer", "reviewer")
builder.add_edge("reviewer", END)

graph = builder.compile()

result = graph.invoke({
    "question": "Write a welcome email for a premium banking customer.",
    "draft": "",
    "review": "",
    "approval": False,
    "final": ""
})

result

In [ ]:
class ReviewState(TypedDict):
    topic: str
    draft: str
    review: str

In [ ]:
builder = StateGraph(ReviewState)

builder.add_node("writer", writer)
builder.add_node("reviewer", reviewer)

builder.add_edge(START, "writer")
builder.add_edge("writer", "reviewer")
builder.add_edge("reviewer", END)

graph = builder.compile()

result = graph.invoke({
    "topic": "Write a welcome email for a premium banking customer.",
    "draft": "",
    "review": ""
})

result

In [32]:
class ApprovalState(TypedDict):
    draft: str
    approval: bool
    final: str

In [33]:
def approval(state):
    if state['approval']:
        state['final'] = state['draft']
    return state

def rewrite(state):
    state['final'] = "Rewrite Required"
    return state

def route(state):
    if state['approval']:
        return "publish"
    else:
        return "rewrite"

In [ ]:
builder = StateGraph(ApprovalState)

builder.add_node("approval", approval)
builder.add_node("rewrite", rewrite)

builder.add_edge(START, "approval")
builder.add_conditional_edges(
    "approval",
    route,
    {
        "publish": END,
        "rewrite": "rewrite",
    }
)
builder.add_edge("rewrite", END)

graph = builder.compile()
    
result = graph.invoke({
    "draft": "Welcome to our premium banking services.",
    "approval": True,
    "final": ""
})